<a href="https://colab.research.google.com/github/takuonakashima/ai-security-workshop/blob/main/MNIST.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt

# ==========================================
# 1. データの前処理と読み込み
# ==========================================
# 画像をテンソルに変換し、正規化します
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,)) # ピクセル値を[-1, 1]の範囲に正規化
])

# MNISTデータセットのダウンロードとロード
train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)

# ==========================================
# 2. モデルの定義 (シンプルなCNN)
# ==========================================
class SimpleCNN(nn.Module):
    def __init__(self):
        super(SimpleCNN, self).__init__()
        # 1チャンネル(白黒)の入力画像(28x28)を受け取り、32個の特徴マップを出力
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=32, kernel_size=3, padding=1)
        self.relu = nn.ReLU()
        self.maxpool = nn.MaxPool2d(kernel_size=2)
        self.flatten = nn.Flatten()
        # 画像サイズが14x14に縮小されるため、32 * 14 * 14の入力次元となる
        self.fc1 = nn.Linear(32 * 14 * 14, 128)
        self.fc2 = nn.Linear(128, 10) # 0〜9の10クラス分類

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.maxpool(x)
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        x = self.fc2(x)
        return x

model = SimpleCNN()

# ==========================================
# 3. 損失関数と最適化手法の設定
# ==========================================
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# GPUが利用可能な場合はGPUを使用する設定（Colabのランタイム設定でT4 GPU等を選ぶと高速化します）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"使用デバイス: {device}")

# ==========================================
# 4. 学習ループ
# ==========================================
epochs = 3 # 演習用のため3エポックで短く設定しています
print("\n--- 学習を開始します ---")
for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()      # 勾配の初期化
        outputs = model(images)    # 順伝播（予測）
        loss = criterion(outputs, labels) # 損失の計算
        loss.backward()            # 逆伝播（勾配の計算）
        optimizer.step()           # パラメータの更新

        running_loss += loss.item()

    print(f"Epoch [{epoch+1}/{epochs}], Loss: {running_loss/len(train_loader):.4f}")

# ==========================================
# 5. モデルの評価（テストデータでの精度確認）
# ==========================================
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"\n--- 学習完了 ---")
print(f"テストデータでの正解率: {accuracy:.2f}%")

# ==========================================
# 6. 推論結果の一部を視覚化
# ==========================================
# テストデータから1バッチ取得
dataiter = iter(test_loader)
images, labels = next(dataiter)
images_cpu, labels_cpu = images.to(device), labels.to(device)

# 予測
outputs = model(images_cpu)
_, predicted = torch.max(outputs, 1)

# 最初の6個の画像と予測結果を表示
fig = plt.figure(figsize=(10, 4))
for i in range(6):
    ax = fig.add_subplot(1, 6, i+1, xticks=[], yticks=[])
    # 画像の正規化を元に戻して表示
    img = images[i].numpy().squeeze()
    img = (img * 0.5) + 0.5
    ax.imshow(img, cmap='gray')
    ax.set_title(f"Pred: {predicted[i].item()}\nTrue: {labels[i].item()}")
plt.tight_layout()
plt.show()